# 📊 Data Tables

> Data table with pagination, search, sorting, and inline actions

In [ ]:
#| default_exp datatable

In [ ]:
#| export

from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
from fasthtml.common import A, Button as FhButton, I, Span
from fh_matui.foundations import normalize_tokens, stringify, VEnum, dedupe_preserve_order
from fh_matui.core import *
from nbdev.showdoc import show_doc
from fh_matui.components import *

## 🎯 Overview

| Category | Components | Purpose |
|----------|------------|---------|
| 📋 Table | `DataTable` | Paginated data table with search, sort, and actions |
| 🔧 Resource | `DataTableResource` | Zero-boilerplate class to wire everything together |

> 💡 **Note**: Form components (`FormField`, `FormModal`, `FormGrid`) are in the Components module. Import via `from fh_matui.components import *`

---

## 🏗️ Architecture

```
┌─────────────────────────────────────────────────────────┐
│                   DataTableResource                      │
├─────────────────────────────────────────────────────────┤
│  Auto-registers routes:                                  │
│  ├─ GET /resource       → DataTable (list view)         │
│  ├─ GET /resource/action→ FormModal (create/edit)       │
│  └─ POST /resource/save → Save handler (insert/update)  │
├─────────────────────────────────────────────────────────┤
│  DataTable                                              │
│  ├─ Pagination controls (page size, prev/next)          │
│  ├─ Search input (debounced, HTMX)                      │
│  ├─ Sortable column headers                             │
│  └─ Row action menus (edit, delete, custom)             │
├─────────────────────────────────────────────────────────┤
│  FormModal (from components)                            │
│  ├─ Auto-generates fields from column config            │
│  └─ HTMX submit to save endpoint                        │
└─────────────────────────────────────────────────────────┘
```

---

## 📚 Quick Reference

```python
# Complete CRUD in 5 lines
resource = DataTableResource(
    app=app,
    name='products',
    data_source=lambda: db.products(),
    columns=[{'key': 'name', 'label': 'Name'}, {'key': 'price', 'label': 'Price'}]
)
```

---

## 🤔 DataTable vs DataTableResource: When to Use Which?

> **TL;DR:** `DataTable` = UI only, `DataTableResource` = UI + routes + CRUD + forms

### Quick Comparison

| Aspect | `DataTable` | `DataTableResource` |
|--------|-------------|---------------------|
| **What it is** | Pure UI function | Full-stack class |
| **Returns** | HTML table component | Auto-registers 3 routes |
| **Data handling** | You provide pre-paginated data | Handles pagination internally |
| **Routes** | ❌ You write them manually | ✅ Auto-generated |
| **Forms** | ❌ You build them manually | ✅ Auto-generated from columns |
| **CRUD operations** | ❌ You implement handlers | ✅ Built-in with hooks |
| **Use case** | Custom/complex tables | Standard CRUD tables |
| **Lines of code** | ~50-100 lines | ~15 lines |

---

### 📊 DataTable: The Building Block

`DataTable` is a **pure UI component** that renders a paginated table. It doesn't know anything about your database or routes—you handle all that.

```python
# You must write the route handler yourself
@rt("/my-products")
def products_table(req):
    # 1. Extract pagination params
    search, page, page_size = extract_params(req)
    
    # 2. Query your database
    data = db.query(f"SELECT * FROM products LIMIT {page_size} OFFSET {(page-1)*page_size}")
    total = db.query("SELECT COUNT(*) FROM products")[0]
    
    # 3. Return the UI component
    return DataTable(
        data=data,
        total=total,
        page=page,
        page_size=page_size,
        columns=[{"key": "name", "label": "Name"}],
        base_route="/my-products"
    )

# You also need separate routes for create/edit/delete forms...
```

**Use DataTable when:**
- You need custom pagination logic
- You want to embed a table inside a larger page
- You're building something non-standard
- You already have route handlers

---

### 🔧 DataTableResource: The Complete Solution

`DataTableResource` is a **high-level class** that uses `DataTable` internally but also auto-registers routes, generates forms, and handles CRUD operations.

```python
# One object replaces ~100 lines of code
products = DataTableResource(
    app=app,
    base_route="/products",
    columns=[{"key": "name", "label": "Name", "form": {"type": "text"}}],
    get_all=lambda: db.products(),
    get_by_id=lambda id: db.products[id],
    create=lambda data: db.products.insert(data),
    update=lambda id, data: db.products.update(id, data),
    delete=lambda id: db.products.delete(id),
    title="Products"
)

# That's it! Routes are auto-registered:
# GET  /products        → Table view
# GET  /products/action → Create/Edit/View modals
# POST /products/save   → Save handler
```

**Use DataTableResource when:**
- You want standard CRUD functionality
- You want auto-generated forms from column config
- You need hooks for business logic (e.g., API calls)
- You want minimal boilerplate

---

### 🎯 Decision Flowchart

```
Do you need a CRUD table with forms?
├── YES → Use DataTableResource
│         (auto routes, forms, hooks)
│
└── NO → Do you need just a data display?
         ├── YES → Use DataTable
         │         (pure UI, you handle routes)
         │
         └── NO → Consider a custom solution
```

In [ ]:
#| hide
#| eval: false

from fasthtml.jupyter import *
from IPython.display import HTML, Markdown, Image
import socket
import time
import subprocess

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        # Find process using the port
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        
        if result.stdout:
            # Extract PID from netstat output
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"✓ Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"⚠ Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=3333, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find an available port in range {start_port}-{start_port+max_attempts}")

# Stop existing server if running
if 'server' in globals(): 
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

# Try to kill any process on preferred port, then find available port
preferred_port = 5555
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app = FastHTML(hdrs=MatTheme.blue.headers(title="fastmaterial", mode="dark"))
rt = app.route

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"✓ Server running on port {port}")
except Exception as e:
    print(f"✗ Failed to start server: {e}")
    raise

✓ Server running on port 5555


## 📋 DataTable Wrapper

| Component | Purpose |
|-----------|---------|
| `DataTable` | Paginated table with search, sort, and row actions |
| `table_state_from_request` | Extract pagination/search state from request |
| `_action_menu` | Dropdown menu for row actions |

**Features:** HTMX-powered pagination, debounced search, sortable columns, action menus

---

### How It Works

The wrapper expects **pre-paginated data** from your backend. Your SQL backend provides:
- `data`: Current page rows (via `LIMIT/OFFSET`)
- `total`: Total record count (via `COUNT(*)`)

```python
@rt("/my-table")
def my_table(req):
    search, page, page_size = extract_params(req)
    data = db.query(f"SELECT * FROM items LIMIT {page_size} OFFSET {(page-1)*page_size}")
    total = db.query("SELECT COUNT(*) FROM items")
    
    return DataTable(
        data=data,
        total=total,
        page=page,
        page_size=page_size,
        search=search,
        columns=[{"key": "name", "label": "Name"}],
        crud_ops={"create": True, "update": True, "delete": True},
        base_route="/my-table"
    )
```

---

### DataTable Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `data` | `list[dict]` | Current page rows (pre-paginated by backend) |
| `total` | `int` | Total record count (for pagination math) |
| `page` | `int` | Current page number (1-indexed) |
| `page_size` | `int` | Records per page |
| `search` | `str` | Current search term |
| `columns` | `list[dict]` | Column configs: `[{"key": "name", "label": "Name", "searchable": True}]` |
| `crud_ops` | `dict` | Enabled operations: `{"create": True, "update": True, "delete": True}` |
| `base_route` | `str` | Base URL for HTMX requests (e.g., `/crud-products`) |
| `row_id_field` | `str` | Field name for row ID (default: `"id"`) |
| `title` | `str` | Table card header title |
| `container_id` | `str` | HTML id for HTMX targeting (auto-generated if None) |
| `page_sizes` | `list` | Page size options (default: `[5, 10, 20, 50]`) |
| `search_placeholder` | `str` | Placeholder for search input |
| `create_label` | `str` | Label for create button |
| `empty_message` | `str` | Message when no records found |

In [ ]:
#| export
#| code-fold: true

from math import ceil
from urllib.parse import urlencode
from typing import Callable, Optional, Any
from dataclasses import asdict, is_dataclass

# Default page size options
PAGE_SIZES = [5, 10, 20, 50]


def _to_dict(obj: Any) -> dict:
    """
    Convert any record type to dict.
    
    Handles:
    - dict -> pass through
    - dataclass -> asdict()
    - namedtuple -> _asdict()
    - Pydantic model -> model_dump() or dict()
    - ORM/object with __dict__ -> vars() filtered
    - dict-like with keys -> dict()
    - None -> {}
    """
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    # Dataclass
    if is_dataclass(obj) and not isinstance(obj, type):
        return asdict(obj)
    # Namedtuple
    if hasattr(obj, '_asdict'):
        return obj._asdict()
    # Pydantic v2
    if hasattr(obj, 'model_dump'):
        return obj.model_dump()
    # Pydantic v1
    if hasattr(obj, 'dict') and callable(getattr(obj, 'dict')):
        return obj.dict()
    # ORM-style objects with __dict__
    if hasattr(obj, "__dict__"):
        return {k: v for k, v in obj.__dict__.items() if not k.startswith("_")}
    # Dict-like
    if hasattr(obj, 'keys'):
        return dict(obj)
    return {}


def _safe_int(value, default):
    """Safely convert to positive int or return default."""
    try:
        number = int(value)
        return number if number > 0 else default
    except (TypeError, ValueError):
        return default

def table_state_from_request(req, page_sizes=None):
    """
    Extract pagination state from request query params.
    
    Returns dict with: search, page, page_size
    """
    page_sizes = page_sizes or PAGE_SIZES
    params = getattr(req, "query_params", {})
    getter = params.get if hasattr(params, "get") else (lambda key, default=None: params[key] if key in params else default)
    
    search = (getter("search", "") or "").strip()
    page = _safe_int(getter("page", 1), 1)
    page_size = _safe_int(getter("page_size", 10), 10)
    
    if page_size not in page_sizes:
        page_size = page_sizes[0] if page_sizes else 10
    
    return {"search": search, "page": page, "page_size": page_size}


def _page_size_select(current_size: int, search: str, base_route: str, container_id: str, page_sizes: list):
    """Build page size dropdown selector."""
    menu_id = f"{container_id}-page-size-menu"
    options = []
    for size in page_sizes:
        params = urlencode({"search": search, "page_size": size, "page": 1})
        option_cls = "active" if size == current_size else None
        options.append(
            Li(
                f"Show {size}",
                hx_get=f"{base_route}?{params}",
                hx_target=f"#{container_id}",
                hx_push_url="true",
                cls=option_cls
            )
        )
    return Button(
        Span(f"Show {current_size}", cls="small-text grey-text"),
        Icon("arrow_drop_down", cls="small grey-text"),
        Menu(*options, cls="border", id=menu_id),
        cls="transparent small",
        data_ui=f"#{menu_id}"
    )


def _action_menu(
    row: dict,
    row_id: Any,
    crud_ops: dict,
    crud_enabled: dict,
    base_route: str,
    search: str,
    page: int,
    page_size: int,
    container_id: str,
    feedback_id: str
):
    """Build per-row action menu based on enabled CRUD operations."""
    menu_id = f"crud-actions-{row_id}"
    base_query = {"id": row_id, "search": search, "page": page, "page_size": page_size}
    
    def action_item(label: str, icon: str, action: str, confirm: str = None):
        query = urlencode({**base_query, "action": action})
        attrs = {
            "hx_get": f"{base_route}/action?{query}",
            "hx_target": f"#{feedback_id}",
            "hx_swap": "outerHTML"
        }
        if confirm:
            attrs["hx_confirm"] = confirm
        return Li(
            A(
                Icon(icon, cls="tiny"),
                Span(label, cls="max"),
                cls="row middle-align",
                **attrs
            )
        )
    
    items = []
    # View is conditional (read operation) - use crud_enabled for callable support
    if crud_enabled.get("view", True):
        items.append(action_item("View", "visibility", "view"))
    
    if crud_enabled.get("update", False):
        items.append(action_item("Edit", "edit", "edit"))
    
    if crud_enabled.get("delete", False):
        items.append(action_item("Delete", "delete", "delete", confirm="Delete this record?"))
    
    # Custom actions
    for custom_action in crud_ops.get("custom_actions", []):
        # Check per-row condition if provided
        condition = custom_action.get("condition")
        if condition and not condition(row):
            continue
        
        items.append(action_item(
            label=custom_action["label"],
            icon=custom_action["icon"],
            action=custom_action["name"],
            confirm=custom_action.get("confirm")
        ))
    
    return Div(
        Button(
            Icon("more_vert"),
            cls=(ButtonT.text, "circle"),
            data_ui=f"#{menu_id}",
            title="Row actions"
        ),
        Menu(*items, id=menu_id),
        cls="relative"
    )


def DataTable(
    data: list[dict],
    total: int,
    page: int = 1,
    page_size: int = 10,
    search: str = '',
    columns: list[dict] = None,
    crud_ops: dict = None,
    crud_enabled: dict = None,
    base_route: str = '',
    row_id_field: str = 'id',
    title: str = 'Records',
    container_id: str = None,
    page_sizes: list = None,
    search_placeholder: str = 'Search...',
    create_label: str = 'New Record',
    empty_message: str = 'No records match the current filters.'
):
    "Generic data table with server-side pagination, search, and row actions."
    # Defaults
    crud_ops = crud_ops or {"create": False, "update": False, "delete": False}
    crud_enabled = crud_enabled or {k: bool(v) for k, v in crud_ops.items() if k != 'custom_actions'}
    page_sizes = page_sizes or PAGE_SIZES
    container_id = container_id or f"crud-table-{base_route.replace('/', '-').strip('-')}"
    feedback_id = f"{container_id}-feedback"
    
    # Auto-convert data records to dicts (supports dataclass, namedtuple, Pydantic, ORM)
    data = [_to_dict(r) for r in data]
    
    # Auto-generate columns from first data row if not provided
    if columns is None and data:
        columns = [{"key": k, "label": k.replace("_", " ").title()} for k in data[0].keys()]
    columns = columns or []
    
    # Calculate pagination metadata
    total_pages = max(1, ceil(total / page_size)) if total > 0 else 1
    page = min(max(1, page), total_pages)
    start_index = (page - 1) * page_size + 1 if total else 0
    end_index = min(start_index + page_size - 1, total) if total else 0
    summary = f"{start_index}-{end_index} of {total} records" if total else "No matching records"
    base_query = urlencode({"search": search, "page_size": page_size})
    
    # Build table header keys and labels
    header_keys = [col["key"] for col in columns] + ["actions"]
    header_labels = [col.get("label", col["key"]) for col in columns] + [""]
    
    # Build table rows
    table_rows = []
    for row in data:
        row_id = row.get(row_id_field)
        row_dict = {}
        
        for col in columns:
            key = col["key"]
            value = row.get(key, "")
            renderer = col.get("renderer")
            
            if renderer and callable(renderer):
                row_dict[key] = renderer(value, row)
            else:
                row_dict[key] = value
        
        # Add actions column
        row_dict["actions"] = _action_menu(
            row=row,
            row_id=row_id,
            crud_ops=crud_ops,
            crud_enabled=crud_enabled,
            base_route=base_route,
            search=search,
            page=page,
            page_size=page_size,
            container_id=container_id,
            feedback_id=feedback_id
        )
        table_rows.append(row_dict)
    
    # Create button - use crud_enabled for callable support
    create_button = None
    if crud_enabled.get("create", False):
        query = urlencode({"action": "create", "search": search, "page": page, "page_size": page_size})
        create_button = Button(
            Icon("add"),
            create_label,
            hx_get=f"{base_route}/action?{query}",
            hx_target=f"#{feedback_id}",
            hx_swap="outerHTML",
            cls=ButtonT.primary
        )
    
    # Pagination controls
    pagination = None
    if total_pages > 1:
        prev_params = urlencode({"search": search, "page": max(1, page - 1), "page_size": page_size})
        next_params = urlencode({"search": search, "page": min(total_pages, page + 1), "page_size": page_size})
        
        pagination = Div(
            Button(
                Icon("navigate_before"),
                hx_get=f"{base_route}?{prev_params}",
                hx_target=f"#{container_id}",
                hx_push_url="true",
                disabled=page == 1,
                cls="circle"
            ),
            Span(f"Page {page} of {total_pages}", cls="middle"),
            Button(
                Icon("navigate_next"),
                hx_get=f"{base_route}?{next_params}",
                hx_target=f"#{container_id}",
                hx_push_url="true",
                disabled=page == total_pages,
                cls="circle"
            ),
            cls="row center-align"
        )
    
    # Search form
    search_form = Form(
        Div(
            Icon("search", cls="small"),
            Input(
                type="search",
                name="search",
                value=search,
                placeholder=search_placeholder,
                hx_get=base_route,
                hx_trigger="input changed delay:300ms, search",
                hx_target=f"#{container_id}",
                hx_push_url="true",
                hx_include="[name='page_size']"
            ),
            cls="field prefix"
        ),
        Input(type="hidden", name="page_size", value=page_size),
        cls="max"
    )
    
    # Build table with proper Thead/Tbody structure
    data_table = Table(
        Thead(Tr(*[Th(label) for label in header_labels])),
        Tbody(*[
            Tr(*[Td(row_dict.get(key, '')) for key in header_keys])
            for row_dict in table_rows
        ]),
        cls="border"
    ) if data else Div(empty_message, cls="center-align padding")
    
    # Footer section with page size selector and pagination
    footer = None
    if total > 0:
        page_info_row = Div(
            _page_size_select(page_size, search, base_route, container_id, page_sizes),
            Span(summary, cls="small-text grey-text"),
            cls="row"
        )
        if pagination:
            # Two-row layout: info on top, centered pagination below
            footer = Div(
                page_info_row,
                Div(pagination, cls="row center-align"),
            )
        else:
            # Single row when no pagination needed
            footer = page_info_row
    
    return Article(
        Div(id=feedback_id),  # Feedback container for modals/toasts
        Div(
            search_form,
            create_button,
            cls="row"
        ),
        data_table,
        footer,
        id=container_id,
        hx_trigger=f"{container_id}-refresh from:body",
        hx_get=f"{base_route}?{base_query}",
        hx_target=f"#{container_id}",
        hx_swap="outerHTML"
    )

---

## 🎯 CrudContext - Enhanced Hook Context

> **Rich context object for custom CRUD hooks with external API integration support**

The `CrudContext` dataclass provides complete access to request state, user info, database, and record data for implementing complex business logic in hooks.

### 📦 Why CrudContext?

**Before (simple hooks):**
```python
def on_before_create(data):
    data['created_at'] = datetime.now()
    return data  # Limited to data manipulation
```

**After (with CrudContext):**
```python
async def on_create(ctx: CrudContext) -> dict:
    # Access user info
    user_id = ctx.user['user_id']
    
    # Call external API
    api = QuilttClient()
    response = await api.create_connection(
        institution=ctx.record['institution_name'],
        user_id=user_id
    )
    
    # Enrich record with API response
    ctx.record['connection_id'] = response['id']
    ctx.record['status'] = 'pending'
    
    return ctx.record  # DataTableResource will insert this
```

---

### 🏗️ Architecture

```
┌─────────────────────────────────────────────────────────┐
│                   DataTableResource                      │
├─────────────────────────────────────────────────────────┤
│  Enhanced CRUD Hooks (with CrudContext)                 │
│  ├─ on_create(ctx) → Return modified record dict        │
│  ├─ on_update(ctx) → Return modified record dict        │
│  └─ on_delete(ctx) → Perform custom deletion logic      │
├─────────────────────────────────────────────────────────┤
│  CrudContext provides:                                  │
│  ├─ request          → Full Starlette request           │
│  ├─ user             → request.state.user               │
│  ├─ db               → request.state.tenant_db          │
│  ├─ tbl              → request.state.tables[table_name] │
│  ├─ record           → Form data dict                   │
│  └─ record_id        → ID for update/delete             │
├─────────────────────────────────────────────────────────┤
│  Features:                                              │
│  ✅ Async/sync hook support                             │
│  ✅ Auto-refresh via HX-Trigger                         │
│  ✅ Toast notifications                                 │
│  ✅ External API integration                            │
│  ✅ Backward compatible with old hooks                  │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
#| export

import asyncio
import logging
from dataclasses import dataclass
from typing import Callable, Any, Optional
from starlette.responses import HTMLResponse

logger = logging.getLogger(__name__)

@dataclass
class CrudContext:
    """
    🎯 Context object passed to enhanced CRUD operation hooks.
    
    Provides rich access to request state, user info, database, and record data.
    Perfect for implementing complex business logic and external API integration.
    
    ## 📦 Fields
    
    - `request`: Full Starlette request object (headers, query params, session, state)
    - `user`: Current user dict from `request.state.user` (if available)
    - `db`: Database instance from `request.state.tenant_db` (if available)
    - `tbl`: Table instance from `request.state.tables[table_name]` (if available)
    - `record`: Form data dict with field values
    - `record_id`: Record ID for update/delete operations (None for create)
    
    ## 💡 Usage in Hooks
    
    ### Example: Create with External API
    ```python
    async def quiltt_create_connection(ctx: CrudContext) -> dict:
        # Access user info
        user_id = ctx.user['user_id']
        
        # Call external API
        api = QuilttClient()
        response = await api.create_connection(
            institution=ctx.record['institution_name'],
            user_id=user_id
        )
        
        # Enrich record with API response
        ctx.record['connection_id'] = response['id']
        ctx.record['account_id'] = response['account_id']
        ctx.record['status'] = 'pending'
        
        return ctx.record  # DataTableResource will insert this
    
    DataTableResource(
        ...,
        on_create=quiltt_create_connection  # 🆕 Enhanced hook
    )
    ```
    
    ### Example: Soft Delete
    ```python
    def soft_delete_budget(ctx: CrudContext) -> None:
        # Access table directly
        ctx.tbl.update({
            'id': ctx.record_id,
            'is_deleted': True,
            'deleted_at': datetime.now().isoformat(),
            'deleted_by': ctx.user['user_id']
        })
        # No return needed for delete hooks
    
    DataTableResource(
        ...,
        on_delete=soft_delete_budget,  # 🆕 Custom delete logic
        get_table=lambda req: req.state.tables['budgets']
    )
    ```
    
    ### Example: Update with Sync
    ```python
    async def sync_transaction_update(ctx: CrudContext) -> dict:
        # Update external API first
        api = TransactionAPI()
        await api.update_transaction(
            transaction_id=ctx.record_id,
            data=ctx.record
        )
        
        # Add sync timestamp
        ctx.record['last_synced'] = datetime.now().isoformat()
        return ctx.record
    ```
    """
    request: Any                     # Full Starlette request
    user: Optional[dict] = None      # request.state.user (if available)
    db: Optional[Any] = None         # request.state.tenant_db (if available)
    tbl: Optional[Any] = None        # request.state.tables[table_name] (if available)
    record: dict = None              # Form data dict
    record_id: Optional[Any] = None  # ID for update/delete (None for create)
    feedback_id: Optional[str] = None  # Target div ID for HTMX swap (for override handlers)

In [ ]:
show_doc(CrudContext)

---

[source](https://github.com/abhisheksreesaila/fh-matui/blob/master/fh_matui/datatable.py#L372){target="_blank" style="float:right; font-size:smaller"}

### CrudContext

>      CrudContext (request:Any, user:Optional[dict]=None,
>                   db:Optional[Any]=None, tbl:Optional[Any]=None,
>                   record:dict=None, record_id:Optional[Any]=None,
>                   feedback_id:Optional[str]=None)

*🎯 Context object passed to enhanced CRUD operation hooks.*

Provides rich access to request state, user info, database, and record data.
Perfect for implementing complex business logic and external API integration.

## 📦 Fields

- `request`: Full Starlette request object (headers, query params, session, state)
- `user`: Current user dict from `request.state.user` (if available)
- `db`: Database instance from `request.state.tenant_db` (if available)
- `tbl`: Table instance from `request.state.tables[table_name]` (if available)
- `record`: Form data dict with field values
- `record_id`: Record ID for update/delete operations (None for create)

## 💡 Usage in Hooks

### Example: Create with External API
```python
async def quiltt_create_connection(ctx: CrudContext) -> dict:
    # Access user info
    user_id = ctx.user['user_id']

    # Call external API
    api = QuilttClient()
    response = await api.create_connection(
        institution=ctx.record['institution_name'],
        user_id=user_id
    )

    # Enrich record with API response
    ctx.record['connection_id'] = response['id']
    ctx.record['account_id'] = response['account_id']
    ctx.record['status'] = 'pending'

    return ctx.record  # DataTableResource will insert this

DataTableResource(
    ...,
    on_create=quiltt_create_connection  # 🆕 Enhanced hook
)
```

### Example: Soft Delete
```python
def soft_delete_budget(ctx: CrudContext) -> None:
    # Access table directly
    ctx.tbl.update({
        'id': ctx.record_id,
        'is_deleted': True,
        'deleted_at': datetime.now().isoformat(),
        'deleted_by': ctx.user['user_id']
    })
    # No return needed for delete hooks

DataTableResource(
    ...,
    on_delete=soft_delete_budget,  # 🆕 Custom delete logic
    get_table=lambda req: req.state.tables['budgets']
)
```

### Example: Update with Sync
```python
async def sync_transaction_update(ctx: CrudContext) -> dict:
    # Update external API first
    api = TransactionAPI()
    await api.update_transaction(
        transaction_id=ctx.record_id,
        data=ctx.record
    )

    # Add sync timestamp
    ctx.record['last_synced'] = datetime.now().isoformat()
    return ctx.record
```

## 🔧 DataTableResource

| Component | Purpose |
|-----------|---------|
| `DataTableResource` | High-level class that auto-registers table + forms + routes |

**Features:** Auto-registers routes, handles pagination/search/save, **all callbacks receive request**, **async hooks with CrudContext**, **auto-refresh via HX-Trigger**, **layout wrapper for full-page responses**

---

### DataTableResource Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `app` | `FastHTML` | App instance to register routes |
| `base_route` | `str` | Base URL path (e.g., `/products`) |
| `columns` | `list[dict]` | Column config (same as DataTable) |
| `get_all` | `Callable[[Request], list]` | `(req) -> list` of all records |
| `get_by_id` | `Callable[[Request, Any], Any]` | `(req, id) -> record` or None |
| `create` | `Callable[[Request, dict], Any]` | `(req, data) -> record` |
| `update` | `Callable[[Request, Any, dict], Any]` | `(req, id, data) -> record` |
| `delete` | `Callable[[Request, Any], bool]` | `(req, id) -> bool` |
| `title` | `str` | Display title for table |
| `layout_wrapper` | `Callable[[FT, Request], FT]` | Wrap full-page responses in app layout |

> 💡 **All data callbacks receive `request` as first parameter** for multi-tenant support

---

### 🎯 CRUD Hooks (with CrudContext)

| Hook | Signature | Purpose |
|------|-----------|---------|
| `on_create` | `(ctx: CrudContext) -> dict` | Custom create logic with full context |
| `on_update` | `(ctx: CrudContext) -> dict` | Custom update logic with full context |
| `on_delete` | `(ctx: CrudContext) -> None` | Custom delete logic with full context |

**Features:**
- ✅ Async/sync support (hooks can be `async def` or regular `def`)
- ✅ Access to user, db via `CrudContext`
- ✅ Perfect for external API integration (e.g., Quiltt)
- ✅ Auto-refresh table after mutations via HX-Trigger

---

### 🏢 Multi-Tenant Example

In a multi-tenant app where each tenant has their own database (`request.state.tenant_db`):

```python
# Define your data callbacks - all receive request
def get_all_connections(req):
    """Get all connections from tenant's database."""
    tbl = req.state.tenant_db.t.connections
    return tbl(order_by="created_at DESC")

def get_connection_by_id(req, id):
    """Get single connection from tenant's database."""
    tbl = req.state.tenant_db.t.connections
    return tbl[id]

def create_connection(req, data):
    """Create connection in tenant's database."""
    tbl = req.state.tenant_db.t.connections
    return tbl.insert(data)

def update_connection(req, id, data):
    """Update connection in tenant's database."""
    tbl = req.state.tenant_db.t.connections
    data['id'] = id
    return tbl.update(data)

def delete_connection(req, id):
    """Delete connection from tenant's database."""
    tbl = req.state.tenant_db.t.connections
    tbl.delete(id)
    return True

# Create resource - handles all routes automatically
connections = DataTableResource(
    app=app,
    base_route="/connections",
    columns=connection_columns,
    get_all=get_all_connections,
    get_by_id=get_connection_by_id,
    create=create_connection,
    update=update_connection,
    delete=delete_connection,
    title="Bank Connections",
    search_placeholder="Search connections...",
    create_label="Add Connection",
    layout_wrapper=lambda content, req: AppLayout(content, user=req.state.user)
)
```

---

### 📐 Layout Wrapper

For full-page (non-HTMX) requests, wrap the table in your app layout:

```python
def my_layout_wrapper(content, req):
    """Wrap content in app layout with sidebar and navbar."""
    return AppLayout(
        content,
        sidebar_links=get_sidebar_links(),
        nav_bar=NavBar(user=req.state.user),
        title="Dashboard"
    )

DataTableResource(
    ...,
    layout_wrapper=my_layout_wrapper
)
```

**How it works:**
- HTMX requests (`HX-Request: true` header) → Returns partial HTML for table swap
- Full-page requests (no header) → Wraps response with `layout_wrapper(content, req)`

---

### 🚀 Basic Usage Example

```python
# Simple in-memory example (no multi-tenant)
PRODUCTS = [{"id": 1, "name": "Widget", "price": 9.99}]

products = DataTableResource(
    app=app,
    base_route="/products",
    columns=[
        {"key": "name", "label": "Name", "searchable": True},
        {"key": "price", "label": "Price"}
    ],
    get_all=lambda req: PRODUCTS,
    get_by_id=lambda req, id: next((p for p in PRODUCTS if p["id"] == id), None),
    create=lambda req, data: PRODUCTS.append({"id": len(PRODUCTS)+1, **data}),
    update=lambda req, id, data: ...,
    delete=lambda req, id: ...,
    title="Products"
)
```

---

### 🔗 With External API Integration (Async Hook)

```python
async def quiltt_create_connection(ctx: CrudContext) -> dict:
    """Call Quiltt API before storing connection in database."""
    from quiltt_client import QuilttAPI
    
    api = QuilttAPI()
    response = await api.create_connection(
        institution=ctx.record['institution_name'],
        user_id=ctx.user['user_id']
    )
    
    # Enrich record with API response
    ctx.record['connection_id'] = response['id']
    ctx.record['account_id'] = response['account_id']
    ctx.record['status'] = 'pending'
    
    return ctx.record

DataTableResource(
    app=app,
    base_route="/connections",
    columns=connection_columns,
    get_all=lambda req: req.state.tenant_db.t.connections(),
    get_by_id=lambda req, id: req.state.tenant_db.t.connections[id],
    create=lambda req, data: req.state.tenant_db.t.connections.insert(data),
    title="Bank Connections",
    on_create=quiltt_create_connection
)
```

---

## 🎨 Custom Row Actions

> **Add domain-specific actions to the row menu beyond standard View/Edit/Delete**

The `custom_actions` feature allows you to extend the row action menu with custom operations like "Refresh", "Archive", "Clone", "Approve", etc., each with their own handlers, icons, and optional confirmation dialogs.

### 📦 Why Custom Actions?

**Before:** Only View/Edit/Delete hardcoded actions
```python
crud_ops = {"create": True, "update": True, "delete": True}
# ❌ Cannot add "Refresh", "Archive", or domain-specific actions
```

**After:** Extensible action menu with custom handlers
```python
crud_ops = {
    "view": False,  # Can disable View now!
    "update": False,
    "delete": True,
    "custom_actions": [
        {
            "name": "refresh",
            "label": "Refresh Data",
            "icon": "sync",
            "handler": lambda ctx: sync_external_data(ctx.record_id),
            "confirm": None  # No confirmation needed
        },
        {
            "name": "archive",
            "label": "Archive",
            "icon": "archive",
            "handler": handle_archive,
            "confirm": "Archive this record?",
            "condition": lambda row: not row.get('is_archived')  # Only show if not archived
        }
    ]
}
```

---

### 🏗️ Custom Action Structure

Each custom action requires:

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `name` | `str` | ✅ Yes | Internal action identifier (used in routing) |
| `label` | `str` | ✅ Yes | Display text in menu |
| `icon` | `str` | ✅ Yes | Material icon name (e.g., "sync", "archive") |
| `handler` | `Callable` | ✅ Yes | Function that receives `CrudContext` |
| `confirm` | `str` | ❌ Optional | Confirmation dialog message |
| `condition` | `Callable[[dict], bool]` | ❌ Optional | Per-row visibility test |

**Reserved action names:** `view`, `edit`, `delete`, `create` (will raise `ValueError` if used)

---

### 🎯 Handler Signature

Handlers receive a `CrudContext` object with full access to request state:

```python
def handle_refresh(ctx: CrudContext) -> Optional[str]:
    """
    Custom action handler.
    
    Args:
        ctx.request: Full Starlette request
        ctx.user: request.state.user (if available)
        ctx.db: request.state.tenant_db (if available)
        ctx.record: Current row data as dict
        ctx.record_id: ID of the row
    
    Returns:
        Optional success message (defaults to "{label} completed successfully.")
    """
    # Access external API
    api = ExternalAPI(ctx.user['api_token'])
    api.sync_data(ctx.record_id)
    
    # Update record
    ctx.db.t.connections.update({
        'id': ctx.record_id,
        'last_synced': datetime.now().isoformat()
    })
    
    return "Data refreshed successfully!"  # Custom success message
```

**Async handlers are supported:**
```python
async def handle_async_action(ctx: CrudContext) -> str:
    result = await some_async_operation(ctx.record_id)
    return f"Completed: {result}"
```

---

### 📊 Complete Example

```python
# Handler definitions
def refresh_connection(ctx: CrudContext) -> str:
    """Sync data from external API."""
    api = BankAPI(ctx.user['bank_token'])
    response = api.refresh_accounts(ctx.record['connection_id'])
    
    ctx.db.t.connections.update({
        'id': ctx.record_id,
        'last_synced': datetime.now().isoformat(),
        'account_count': response['account_count']
    })
    
    return f"Refreshed {response['account_count']} accounts"

def archive_connection(ctx: CrudContext):
    """Soft delete by archiving."""
    ctx.db.t.connections.update({
        'id': ctx.record_id,
        'is_archived': True,
        'archived_at': datetime.now().isoformat(),
        'archived_by': ctx.user['user_id']
    })
    # Return None = uses default success message

async def clone_connection(ctx: CrudContext) -> str:
    """Create a duplicate connection."""
    new_record = {**ctx.record}
    del new_record['id']
    new_record['name'] = f"{new_record['name']} (Copy)"
    new_record['created_at'] = datetime.now().isoformat()
    
    # Async database operation
    result = await ctx.db.t.connections.insert_async(new_record)
    return f"Cloned as connection #{result['id']}"

# DataTableResource configuration
DataTableResource(
    app=app,
    base_route="/connections",
    columns=connection_columns,
    get_all=lambda req: req.state.tenant_db.t.connections(),
    get_by_id=lambda req, id: req.state.tenant_db.t.connections[id],
    crud_ops={
        "view": False,  # 🆕 Disable View action
        "create": True,
        "update": False,  # No edit needed
        "delete": True,
        "custom_actions": [  # 🆕 Custom actions
            {
                "name": "refresh",
                "label": "Refresh Data",
                "icon": "sync",
                "handler": refresh_connection
            },
            {
                "name": "archive",
                "label": "Archive",
                "icon": "archive",
                "handler": archive_connection,
                "confirm": "Archive this connection?",
                "condition": lambda row: not row.get('is_archived')  # Only show if active
            },
            {
                "name": "clone",
                "label": "Duplicate",
                "icon": "content_copy",
                "handler": clone_connection,
                "confirm": "Create a copy of this connection?"
            }
        ]
    },
    title="Bank Connections"
)
```

---

### 🔧 Per-Row Conditional Actions

Use the `condition` callable to show/hide actions based on row data:

```python
{
    "name": "approve",
    "label": "Approve",
    "icon": "check_circle",
    "handler": handle_approve,
    "condition": lambda row: row.get('status') == 'pending'  # Only show for pending items
}
```

**Common conditions:**
- Status-based: `lambda row: row['status'] == 'active'`
- Role-based: `lambda row: row['owner_id'] == current_user_id` (access via handler)
- Feature flag: `lambda row: row.get('supports_refresh', False)`
- Type-based: `lambda row: row['type'] in ['checking', 'savings']`

---

### 🎭 Future: Full Renderer Override (Escape Hatch)

For complete UI control beyond menu items, a future `row_actions_renderer` parameter will allow replacing the entire action menu:

```python
# 🔮 Future feature (not yet implemented)
def custom_row_actions(row, row_id, ctx):
    """Full control over row action UI."""
    if row['type'] == 'external':
        return Button(
            "Launch Widget",
            onclick="launchExternalWidget()",
            cls="secondary"
        )
    # Fall back to standard menu
    return None

DataTable(
    ...,
    row_actions_renderer=custom_row_actions  # 🔮 Planned
)
```

This would be useful for:
- Non-menu layouts (button groups, custom dropdowns)
- External widget integrations
- Row-specific completely different UIs

**Status:** Documented as planned feature, implement only if real use cases emerge.

---

In [ ]:
#| export

from typing import Callable, Optional, Any, Union
from dataclasses import asdict, is_dataclass
from datetime import datetime
import uuid

def _to_dict(obj: Any) -> dict:
    """Convert dataclass, ORM object, or dict to plain dict."""
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if is_dataclass(obj):
        return asdict(obj)
    # ORM-style objects with __dict__
    if hasattr(obj, "__dict__"):
        return {k: v for k, v in obj.__dict__.items() if not k.startswith("_")}
    return dict(obj)


def _is_htmx_request(req) -> bool:
    """Check if request is an HTMX partial request."""
    headers = getattr(req, 'headers', {})
    return headers.get('HX-Request') == 'true'


class DataTableResource:
    """
    🔧 High-level resource that auto-registers all routes for a data table.
    
    **Features:**
    - All callbacks receive `request` for multi-tenant support
    - Custom CRUD hooks with `CrudContext` for rich business logic
    - Async/sync hook support for external API integration
    - Auto-refresh table via HX-Trigger after mutations
    - Layout wrapper for full-page (non-HTMX) responses
    
    **Auto-registers 3 routes:**
    - `GET {base_route}` → DataTable list view
    - `GET {base_route}/action` → FormModal for create/edit/view/delete
    - `POST {base_route}/save` → Save handler with hooks
    """
    
    def __init__(
        self,
        app,
        base_route: str,
        columns: list[dict],
        # Data callbacks - ALL receive request as first param
        get_all: Callable[[Any], list],                    # (req) -> list
        get_by_id: Callable[[Any, Any], Any],              # (req, id) -> record
        create: Callable[[Any, dict], Any] = None,         # (req, data) -> record
        update: Callable[[Any, Any, dict], Any] = None,    # (req, id, data) -> record
        delete: Callable[[Any, Any], bool] = None,         # (req, id) -> bool
        # Display options
        title: str = "Records",
        row_id_field: str = "id",
        crud_ops: dict = None,
        page_sizes: list = None,
        search_placeholder: str = "Search...",
        create_label: str = "New Record",
        empty_message: str = "No records found.",
        # CRUD Hooks (with CrudContext)
        on_create: Callable[[CrudContext], dict] = None,
        on_update: Callable[[CrudContext], dict] = None,
        on_delete: Callable[[CrudContext], None] = None,
        # Layout wrapper for full-page responses
        layout_wrapper: Callable[[Any, Any], Any] = None,  # (content, req) -> wrapped
        # Custom generators
        id_generator: Callable[[], Any] = None,
        timestamp_fields: dict = None
    ):
        self.app = app
        self.base_route = base_route.rstrip("/")
        self.columns = columns
        self.get_all = get_all
        self.get_by_id = get_by_id
        self.create_fn = create
        self.update_fn = update
        self.delete_fn = delete
        self.title = title
        self.row_id_field = row_id_field
        self.page_sizes = page_sizes or PAGE_SIZES
        self.search_placeholder = search_placeholder
        self.create_label = create_label
        self.empty_message = empty_message
        
        # Determine CRUD ops from provided functions
        if crud_ops is None:
            self.crud_ops = {
                "create": create is not None or on_create is not None,
                "update": update is not None or on_update is not None,
                "delete": delete is not None or on_delete is not None
            }
        else:
            self.crud_ops = crud_ops
            
            # Validate custom_actions if provided
            custom_actions = crud_ops.get("custom_actions", [])
            if custom_actions:
                reserved_names = {"view", "edit", "delete", "create"}
                for action in custom_actions:
                    # Check required fields
                    if "name" not in action:
                        raise ValueError("Custom action missing required 'name' field")
                    if "label" not in action:
                        raise ValueError(f"Custom action '{action['name']}' missing required 'label' field")
                    if "icon" not in action:
                        raise ValueError(f"Custom action '{action['name']}' missing required 'icon' field")
                    if "handler" not in action:
                        raise ValueError(f"Custom action '{action['name']}' missing required 'handler' field")
                    
                    # Check for reserved name collisions
                    if action["name"] in reserved_names:
                        raise ValueError(f"Custom action name '{action['name']}' conflicts with reserved action name")
                    
                    # Validate handler is callable
                    if not callable(action["handler"]):
                        raise ValueError(f"Custom action '{action['name']}' handler must be callable")
        
        # Parse crud_ops for callable overrides vs boolean enable/disable
        # Callable = override handler (action is enabled)
        # True = use default handler (action is enabled)
        # False = action is disabled
        self.crud_overrides = {}
        self.crud_enabled = {}
        for action_name in ['create', 'update', 'delete', 'view']:
            value = self.crud_ops.get(action_name, action_name == 'view')  # view defaults to True
            if callable(value):
                self.crud_overrides[action_name] = value
                self.crud_enabled[action_name] = True  # Callable means action is enabled
            else:
                self.crud_enabled[action_name] = bool(value)
        
        # CRUD hooks
        self.on_create_hook = on_create
        self.on_update_hook = on_update
        self.on_delete_hook = on_delete
        
        # Layout wrapper
        self.layout_wrapper = layout_wrapper
        
        # Generators
        self.id_generator = id_generator
        self.timestamp_fields = timestamp_fields or {}
        
        # Derived IDs
        self.container_id = f"crud-table-{base_route.replace('/', '-').strip('-')}"
        self.feedback_id = f"{self.container_id}-feedback"
        self.modal_id = f"{self.container_id}-modal"
        self.refresh_trigger = f"{self.container_id}-refresh"
        
        # Register routes
        self._register_routes()
    
    async def _call_hook(self, hook, ctx: CrudContext):
        """🔄 Call hook function, handling both sync and async."""
        if hook is None:
            return None
        if asyncio.iscoroutinefunction(hook):
            return await hook(ctx)
        return hook(ctx)
    
    def _build_context(self, req, record: dict = None, record_id: Any = None, include_feedback_id: bool = False) -> CrudContext:
        """🏗️ Build CrudContext from request."""
        user = None
        db = None
        tbl = None
        
        # Extract user from request.state if available
        try:
            if hasattr(req, 'state') and hasattr(req.state, 'user'):
                user = req.state.user
        except AttributeError:
            pass
        
        # Extract db from request.state if available
        try:
            if hasattr(req, 'state') and hasattr(req.state, 'tenant_db'):
                db = req.state.tenant_db
        except AttributeError:
            pass
        
        return CrudContext(
            request=req,
            user=user,
            db=db,
            tbl=tbl,
            record=record or {},
            record_id=record_id,
            feedback_id=self.feedback_id if include_feedback_id else None
        )
    
    def _wrap_response(self, content, req):
        """Wrap content with layout_wrapper if not an HTMX request."""
        if self.layout_wrapper and not _is_htmx_request(req):
            return self.layout_wrapper(content, req)
        return content
    
    def _register_routes(self):
        """Register all data table routes with the app."""
        rt = self.app.route
        
        @rt(self.base_route)
        def _table_handler(req):
            return self._handle_table(req)
        
        @rt(f"{self.base_route}/action")
        async def _action_handler(req):
            return await self._handle_action(req)
        
        @rt(f"{self.base_route}/save")
        async def _save_handler(req):
            return await self._handle_save(req)
    
    def _get_filtered_data(self, req) -> list:
        """Get all data from user's callback."""
        all_data = self.get_all(req)
        return [_to_dict(item) for item in all_data]
    
    def _filter_by_search(self, data: list, search: str) -> list:
        """Filter data by search term across searchable columns."""
        if not search:
            return data
        
        needle = search.lower()
        searchable_keys = [
            col["key"] for col in self.columns 
            if col.get("searchable", False)
        ]
        
        if not searchable_keys:
            searchable_keys = [col["key"] for col in self.columns]
        
        return [
            row for row in data
            if any(needle in str(row.get(key, "")).lower() for key in searchable_keys)
        ]
    
    def _paginate(self, data: list, page: int, page_size: int) -> tuple:
        """Paginate data list. Returns (page_rows, total, adjusted_page)."""
        total = len(data)
        total_pages = max(1, ceil(total / page_size))
        page = min(max(1, page), total_pages)
        start = (page - 1) * page_size
        end = start + page_size
        return data[start:end], total, page
    
    def _handle_table(self, req):
        """Handle main table route."""
        state = table_state_from_request(req, page_sizes=self.page_sizes)
        search, page, page_size = state["search"], state["page"], state["page_size"]
        
        data = self._get_filtered_data(req)
        filtered = self._filter_by_search(data, search)
        page_data, total, page = self._paginate(filtered, page, page_size)
        
        table = DataTable(
            data=page_data,
            total=total,
            page=page,
            page_size=page_size,
            search=search,
            columns=self.columns,
            crud_ops=self.crud_ops,
            crud_enabled=self.crud_enabled,
            base_route=self.base_route,
            row_id_field=self.row_id_field,
            title=self.title,
            container_id=self.container_id,
            page_sizes=self.page_sizes,
            search_placeholder=self.search_placeholder,
            create_label=self.create_label,
            empty_message=self.empty_message
        )
        
        # Wrap table in auto-refresh container
        params = urlencode({"search": search, "page": page, "page_size": page_size})
        table_container = Div(
            table,
            id=self.container_id,
            hx_trigger=f"{self.refresh_trigger} from:body",
            hx_get=f"{self.base_route}?{params}",
            hx_target=f"#{self.container_id}",
            hx_swap="outerHTML"
        )
        
        feedback = Div(id=self.feedback_id)
        content = Div(feedback, table_container)
        
        # Wrap with layout if full-page request
        return self._wrap_response(content, req)
    
    async def _handle_action(self, req):
        """Handle action route (view/edit/create/delete) with override support."""
        params = getattr(req, "query_params", {})
        getter = params.get if hasattr(params, "get") else (lambda k, d=None: params[k] if k in params else d)
        
        if getter("dismiss") is not None:
            return Div(id=self.feedback_id)
        
        record_id = getter("id")
        if record_id:
            try:
                record_id = int(record_id)
            except (TypeError, ValueError):
                pass
        
        action = (getter("action", "view") or "view").lower()
        search = getter("search", "") or ""
        page = _safe_int(getter("page", 1), 1)
        page_size = _safe_int(getter("page_size", 10), 10)
        
        return_params = urlencode({"search": search, "page": page, "page_size": page_size})
        cancel_url = f"{self.base_route}/action?dismiss=1"
        save_url = f"{self.base_route}/save?{return_params}"
        
        # Get record for actions that need it (not create)
        record = None
        if record_id:
            raw_record = self.get_by_id(req, record_id)
            record = _to_dict(raw_record) if raw_record else None
        
        # Map action names to internal action names for overrides
        action_map = {'edit': 'update'}  # 'edit' action uses 'update' override
        override_key = action_map.get(action, action)
        
        # Check for CRUD override FIRST (callable in crud_ops)
        if override_key in self.crud_overrides:
            try:
                ctx = self._build_context(req, record=record, record_id=record_id, include_feedback_id=True)
                handler = self.crud_overrides[override_key]
                
                # Execute handler (sync or async)
                if asyncio.iscoroutinefunction(handler):
                    result = await handler(ctx)
                else:
                    result = handler(ctx)
                
                # Handle return value:
                # - FT component: Return directly (full control to user)
                # - str: Wrap in success toast
                # - None: Default success message
                if result is None:
                    return self._success_toast(f"{action.title()} completed successfully.")
                elif isinstance(result, str):
                    return self._success_toast(result)
                else:
                    # FT component - wrap in feedback div
                    return Div(result, id=self.feedback_id)
                    
            except Exception as e:
                logger.error(f"CRUD override '{action}' failed: {e}", exc_info=True)
                return self._error_toast(f"{action.title()} failed: {str(e)}")
        
        # Handle CREATE (default behavior)
        if action == "create":
            if not self.crud_enabled.get("create"):
                return self._error_toast("Create operation not enabled.")
            
            modal = FormModal(
                columns=self.columns,
                mode="create",
                record=None,
                modal_id=self.modal_id,
                title=f"New {self.title.rstrip('s')}",
                save_url=save_url,
                save_target=f"#{self.feedback_id}",
                cancel_url=cancel_url,
                cancel_target=f"#{self.feedback_id}"
            )
            return self._wrap_modal(modal)
        
        # Record required for remaining actions
        if not record:
            return self._error_toast("Record not found.")
        
        # Handle VIEW (default behavior)
        if action == "view":
            modal = FormModal(
                columns=self.columns,
                mode="view",
                record=record,
                modal_id=self.modal_id,
                title=f"View {self.title.rstrip('s')}",
                cancel_url=cancel_url,
                cancel_target=f"#{self.feedback_id}"
            )
            return self._wrap_modal(modal)
        
        # Handle EDIT (default behavior)
        if action == "edit":
            if not self.crud_enabled.get("update"):
                return self._error_toast("Update operation not enabled.")
            
            modal = FormModal(
                columns=self.columns,
                mode="edit",
                record=record,
                modal_id=self.modal_id,
                title=f"Edit {self.title.rstrip('s')}",
                save_url=save_url,
                save_target=f"#{self.feedback_id}",
                cancel_url=cancel_url,
                cancel_target=f"#{self.feedback_id}"
            )
            return self._wrap_modal(modal)
        
        # Handle DELETE (default behavior)
        if action == "delete":
            if not self.crud_enabled.get("delete"):
                return self._error_toast("Delete operation not enabled.")
            
            try:
                ctx = self._build_context(req, record=record, record_id=record_id)
                
                # Use on_delete hook if provided
                if self.on_delete_hook:
                    await self._call_hook(self.on_delete_hook, ctx)
                else:
                    # Default: use delete_fn
                    self.delete_fn(req, record_id)
                
                return self._success_toast("Record deleted successfully.")
            except Exception as e:
                logger.error(f"Delete failed: {e}", exc_info=True)
                return self._error_toast(f"Delete failed: {str(e)}")
        
        # Handle CUSTOM ACTIONS
        custom_actions = self.crud_ops.get("custom_actions", [])
        for custom_action in custom_actions:
            if action == custom_action["name"]:
                try:
                    ctx = self._build_context(req, record=record, record_id=record_id, include_feedback_id=True)
                    handler = custom_action["handler"]
                    
                    # Execute handler (sync or async)
                    if asyncio.iscoroutinefunction(handler):
                        result = await handler(ctx)
                    else:
                        result = handler(ctx)
                    
                    # Handle return value same as overrides
                    if result is None:
                        return self._success_toast(f"{custom_action['label']} completed successfully.")
                    elif isinstance(result, str):
                        return self._success_toast(result)
                    else:
                        return Div(result, id=self.feedback_id)
                    
                except Exception as e:
                    logger.error(f"Custom action '{action}' failed: {e}", exc_info=True)
                    return self._error_toast(f"{custom_action['label']} failed: {str(e)}")
        
        return Div(id=self.feedback_id)
    
    async def _handle_save(self, req):
        """Handle save route (create/update form submission)."""
        try:
            form_data = await req.form()
            record_id = form_data.get(self.row_id_field)
            
            if record_id:
                try:
                    record_id = int(record_id)
                except (TypeError, ValueError):
                    pass
            
            # Build data dict from form
            data = {}
            for col in self.columns:
                key = col["key"]
                if key == self.row_id_field:
                    continue
                
                form_cfg = col.get("form", {})
                if form_cfg.get("hidden"):
                    continue
                
                value = form_data.get(key)
                
                field_type = form_cfg.get("type", "text")
                if field_type == "number" and value:
                    try:
                        value = float(value) if "." in str(value) else int(value)
                    except (TypeError, ValueError):
                        pass
                elif field_type == "checkbox":
                    value = value == "on" or value == "true" or value == True
                
                data[key] = value
            
            # Add timestamps
            now = datetime.now()
            for field, ts_type in self.timestamp_fields.items():
                if ts_type == "updated":
                    data[field] = now
                elif ts_type == "created" and not record_id:
                    data[field] = now
            
            # CREATE or UPDATE
            if record_id:
                # ===== UPDATE =====
                ctx = self._build_context(req, record=data, record_id=record_id)
                
                if self.on_update_hook:
                    data = await self._call_hook(self.on_update_hook, ctx)
                    if data is not None:
                        self.update_fn(req, record_id, data)
                else:
                    self.update_fn(req, record_id, data)
                
                return self._success_toast("Record updated successfully.")
            else:
                # ===== CREATE =====
                # Generate ID if needed
                if self.id_generator and self.row_id_field not in data:
                    data[self.row_id_field] = self.id_generator()
                
                ctx = self._build_context(req, record=data, record_id=None)
                
                if self.on_create_hook:
                    data = await self._call_hook(self.on_create_hook, ctx)
                    if data is not None:
                        self.create_fn(req, data)
                else:
                    self.create_fn(req, data)
                
                return self._success_toast("Record created successfully.")
        
        except Exception as e:
            logger.error(f"Save failed: {e}", exc_info=True)
            return self._error_toast(f"Save failed: {str(e)}")
    
    def _wrap_modal(self, modal):
        """Wrap modal content in feedback div."""
        if isinstance(modal, list):
            return Div(*modal, id=self.feedback_id)
        return Div(modal, id=self.feedback_id)
    
    def _success_toast(self, message: str):
        """✅ Return a success toast with auto-refresh trigger."""
        toast = Toast(
            message,
            variant="success",
            position="top",
            action=A(
                "Dismiss",
                cls="inverse-link",
                hx_get=f"{self.base_route}/action?dismiss=1",
                hx_target=f"#{self.feedback_id}",
                hx_swap="outerHTML"
            ),
            active=True
        )
        
        # Return as HTMLResponse with HX-Trigger for auto-refresh
        from fasthtml.common import to_xml
        html_content = to_xml(Div(toast, id=self.feedback_id))
        response = HTMLResponse(content=html_content)
        response.headers['HX-Trigger'] = self.refresh_trigger
        return response
    
    def _error_toast(self, message: str):
        """❌ Return an error toast (no auto-refresh on errors)."""
        toast = Toast(
            message,
            variant="error",
            position="top",
            action=A(
                "Dismiss",
                cls="inverse-link",
                hx_get=f"{self.base_route}/action?dismiss=1",
                hx_target=f"#{self.feedback_id}",
                hx_swap="outerHTML"
            ),
            active=True
        )
        return Div(toast, id=self.feedback_id)

### DataTableResource Demo

Demonstrates how `DataTableResource` reduces ~100 lines of route handlers to ~15 lines:

In [ ]:
#| code-fold: true
#| eval: false

# --- Mock Data for Demo ---
MOCK_PRODUCTS = [
    {"id": 1, "name": "MacBook Pro 16\"", "category": "Laptops", "price": 2499.00, "stock": 45, "status": "In Stock"},
    {"id": 2, "name": "iPhone 15 Pro", "category": "Phones", "price": 999.00, "stock": 120, "status": "In Stock"},
    {"id": 3, "name": "iPad Air", "category": "Tablets", "price": 599.00, "stock": 0, "status": "Out of Stock"},
    {"id": 4, "name": "AirPods Pro", "category": "Audio", "price": 249.00, "stock": 200, "status": "In Stock"},
    {"id": 5, "name": "Apple Watch Ultra", "category": "Wearables", "price": 799.00, "stock": 30, "status": "Low Stock"},
    {"id": 6, "name": "Magic Keyboard", "category": "Accessories", "price": 299.00, "stock": 85, "status": "In Stock"},
    {"id": 7, "name": "Studio Display", "category": "Monitors", "price": 1599.00, "stock": 12, "status": "Low Stock"},
    {"id": 8, "name": "Mac Mini M2", "category": "Desktops", "price": 599.00, "stock": 60, "status": "In Stock"},
    {"id": 9, "name": "HomePod Mini", "category": "Audio", "price": 99.00, "stock": 150, "status": "In Stock"},
    {"id": 10, "name": "AirTag 4-Pack", "category": "Accessories", "price": 99.00, "stock": 0, "status": "Out of Stock"},
]

# Status chip styles
PRODUCT_STATUS_CLASSES = {
    "In Stock": "chip small success",
    "Low Stock": "chip small warning", 
    "Out of Stock": "chip small error"
}

# Column configuration with renderers and form config
product_columns = [
    {
        "key": "name",
        "label": "Product",
        "searchable": True,
        "renderer": lambda v, row: Strong(v),
        "form": {"type": "text", "required": True}
    },
    {
        "key": "category",
        "label": "Category",
        "searchable": True,
        "form": {
            "type": "select",
            "options": ["Laptops", "Phones", "Tablets", "Audio", "Wearables", "Accessories", "Monitors", "Desktops"]
        }
    },
    {
        "key": "price",
        "label": "Price",
        "renderer": lambda v, row: Span(f"${v:,.2f}", cls="bold"),
        "form": {"type": "number", "min": 0, "step": 0.01}
    },
    {
        "key": "stock",
        "label": "Stock",
        "renderer": lambda v, row: Span(str(v), cls=f"badge {'error' if v == 0 else 'warning' if v < 20 else 'success'}") if isinstance(v, int) else v,
        "form": {"type": "number", "min": 0, "step": 1}
    },
    {
        "key": "status",
        "label": "Status",
        "searchable": True,
        "renderer": lambda v, row: Span(v, cls=PRODUCT_STATUS_CLASSES.get(v, "chip small")),
        "form": {"type": "select", "options": ["In Stock", "Low Stock", "Out of Stock"]}
    }
]

# --- In-memory store (simulates database) ---
DEMO_PRODUCTS = list(MOCK_PRODUCTS)

# All callbacks receive request as first parameter
def demo_get_all(req):
    """Get all products. In multi-tenant app: req.state.tenant_db.t.products()"""
    return DEMO_PRODUCTS

def demo_get_by_id(req, id):
    """Get product by ID. In multi-tenant app: req.state.tenant_db.t.products[id]"""
    return next((p for p in DEMO_PRODUCTS if p["id"] == id), None)

def demo_create(req, data):
    """Create product. In multi-tenant app: req.state.tenant_db.t.products.insert(data)"""
    new_id = max(p["id"] for p in DEMO_PRODUCTS) + 1
    record = {"id": new_id, **data}
    DEMO_PRODUCTS.append(record)
    return record

def demo_update(req, id, data):
    """Update product. In multi-tenant app: tbl.update({'id': id, **data})"""
    for i, p in enumerate(DEMO_PRODUCTS):
        if p["id"] == id:
            DEMO_PRODUCTS[i] = {"id": id, **data}
            return DEMO_PRODUCTS[i]
    return None

def demo_delete(req, id):
    """Delete product. In multi-tenant app: req.state.tenant_db.t.products.delete(id)"""
    global DEMO_PRODUCTS
    DEMO_PRODUCTS = [p for p in DEMO_PRODUCTS if p["id"] != id]
    return True

# --- DataTableResource: One object replaces ~100 lines of route handlers ---
products_resource = DataTableResource(
    app=app,
    base_route="/products",
    columns=product_columns,
    get_all=demo_get_all,
    get_by_id=demo_get_by_id,
    create=demo_create,
    update=demo_update,
    delete=demo_delete,
    title="Products",
    search_placeholder="Search products...",
    create_label="Add Product"
)

# --- Preview helper ---
def ex_products():
    """Preview the DataTableResource table."""
    mock_req = type('MockRequest', (), {'query_params': {}, 'headers': {}})()
    return products_resource._handle_table(mock_req)

preview(ex_products())

In [ ]:
#| hide

import nbdev as nb
nb.nbdev_export()